In [4]:
# Copied from main file
from getpass import getpass
import io
import pandas as pd
import msoffcrypto

file_path =r"C:\Users\danam\OneDrive\Desktop\Bachelorprojekt\Tidligere excel bearbejdninger\Alle Giftlinjen 2018-2022 (anonym) Aldersfordelt - Kopi.xlsx"
password = getpass("Enter Excel password: ")

decrypted = io.BytesIO()
with open(file_path, "rb") as f:
    office = msoffcrypto.OfficeFile(f)
    office.load_key(password=password)
    office.decrypt(decrypted)

# Move back to the start of the stream before reading
decrypted.seek(0)

# Specify the two sheets you want to load
sheet_names = ["Alder<70", "Alder>70"] 
# List the columns you want
keep = [
    '[Indtag tid(gift1)]',
    '[Mand(Patient data)]',
    '[Alder (år)(Patient data)]',
    '[Ingen risiko(Risiko)]',
    '[generisk navn(gift1)]',
    '[navn(gift1)]',
    '[substans type(gift1)]',
    '[Enhed(gift1)]',
    '[Styrke(gift1)]',
    '[antal(gift1)]',
    '[dosis(gift1)]',
    '[generisk navn(gift2)]',
    '[navn(gift2)]',
    '[substans type(gift2)]',
    '[Enhed(gift2)]',
    '[Styrke(gift2)]',
    '[antal(gift2)]',
    '[dosis(gift2)]',
    '[generisk navn(gift3)]',
    '[navn(gift3)]',
    '[substans type(gift3)]',
    '[Enhed(gift3)]',
    '[Styrke(gift3)]',
    '[antal(gift3)]',
    '[dosis(gift3)]',
    '[generisk navn(gift4)]',
    '[navn(gift4)]',
    '[substans type(gift4)]',
    '[Enhed(gift4)]',
    '[Styrke(gift4)]',
    '[antal(gift4)]',
    '[dosis(gift4)]',  
    '[generisk navn(gift5)]',
    '[navn(gift5)]',
    '[substans type(gift5)]',
    '[Enhed(gift5)]',
    '[Styrke(gift5)]',
    '[antal(gift5)]',
    '[dosis(gift5)]',
    '[generisk navn(gift6)]',
    '[navn(gift6)]',
    '[substans type(gift6)]',
    '[Enhed(gift6)]',
    '[Styrke(gift6)]',
    '[antal(gift6)]',
    '[Dosis(gift6)]',
    '[generisk navn(gift7)]',
    '[navn(gift7)]',
    '[substans type(gift7)]',
    '[Enhed(gift7)]',
    '[Styrke(gift7)]',
    '[antal(gift7)]',
    '[Dosis(gift7)]',
    '[generisk navn(gift8)]',
    '[navn(gift8)]',
    '[substans type(gift8)]',
    '[Enhed(gift8)]',
    '[Styrke(gift8)]',
    '[antal(gift8)]',
    '[Dosis(gift8)]',
    '[generisk navn(gift9)]',
    '[navn(gift9)]',
    '[substans type(gift9)]',
    '[Enhed(gift9)]',
    '[Styrke(gift9)]',
    '[antal(gift9)]',
    '[Dosis(gift9)]',
    '[generisk navn(gift10)]',
    '[navn(gift10)]',
    '[substans type(gift10)]',
    '[Enhed(gift10)]',
    '[Styrke(gift10)]',
    '[antal(gift10)]',
    '[Dosis(gift10)]',
    '[generisk navn(gift11)]',
    '[navn(gift11)]',
    '[substans type(gift11)]',
    '[Enhed(gift11)]',
    '[Styrke(gift11)]',
    '[antal(gift11)]',
    '[Dosis(gift11)]',
    '[generisk navn(gift12)]',
    '[navn(gift12)]',
    '[substans type(gift12)]',
    '[Enhed(gift12)]',
    '[Styrke(gift12)]',
    '[antal(gift12)]',
    '[Dosis(gift12)]',
    '[generisk navn(gift13)]',
    '[navn(gift13)]',
    '[substans type(gift13)]',
    '[Enhed(gift13)]',
    '[Styrke(gift13)]',
    '[antal(gift13)]',
    '[Dosis(gift13)]',
    '[generisk navn(gift14)]',
    '[navn(gift14)]',
    '[substans type(gift14)]',
    '[Enhed(gift14)]',
    '[Styrke(gift14)]',
    '[antal(gift14)]',
    '[Dosis(gift14)]',
    '[generisk navn(gift15)]',
    '[navn(gift15)]',
    '[substans type(gift15)]',
    '[Enhed(gift15)]',
    '[Styrke(gift15)]',
    '[antal(gift15)]',
    '[Dosis(gift15)]',
    '[Ingen(Anbefalet obs)]',
    '[Akut forgiftning(Forespørgselsart)]',
    '[Søg Hospital(hospital)]',
    '[Borger(Spørger1)]',
    '[Søg postnummer(postalcode)]',
    '[Søg By(postalcode)]',
    '[Råd(Tekst)]',
    '[Today(Oprettelse)]',
    '[Anamnese(Tekst)]',
    '[Psykiatriske sygdomme(Komorbiditet)]',
    '[Ulykke/uheld(Årsag)]',
    '[Leg(Årsag)]',
    '[Suicidal/Affekt(Årsag)]',
    '[Misbrug(Årsag)]',
    '[Forveksling(Årsag)]',
    '[Omhældning(Årsag)]',
    '[Fejldosering(Årsag)]',
    '[Levnedsmiddel(Årsag)]',
]

dfs = pd.read_excel(
    decrypted,
    sheet_name=sheet_names,
    usecols=keep
)

# All column names are now lowercase:
for sheet in sheet_names:
    dfs[sheet].columns = [c.lower() for c in dfs[sheet].columns]


young = dfs[sheet_names[0]]
old = dfs[sheet_names[1]]


Detect duplicate cases according to the *reference-row* rule.

    1.  Two rows can only belong to the same case if they already agree on
        the hard keys stored in KEY_COLS.

    2.  Inside every such key-group the rows are split whenever the gap
        between consecutive '[today(oprettelse)]' values exceeds 7 days.

    3.  In each resulting time-segment we repeatedly
          • pick as REFERENCE the row that contains the largest number of
            non-missing values in the eight Årsag columns (ties → first row),
          • compare every remaining row with that reference:

                – if the candidate is NaN in a column ⇒ that column matches
                – if the candidate has a value while the reference is NaN
                  ⇒ ***mismatch***
                – if both hold values they must be identical, otherwise
                  ⇒ ***mismatch***

          • every candidate with *no* mismatching columns joins the current
            duplicate cluster; the others are processed again with a new
            reference.  Only clusters that end up with ≥ 2 rows are kept.

    4.  The returned frame contains all rows that belong to such clusters and
        adds the columns
            is_duplicate  (always True)
            dup_case_id   (consecutive integer id per duplicate case)

In [11]:
# =================================================================
# 3.  duplicate-detection
# =================================================================
def find_duplicates(df):
    """
    Detect duplicate cases according to the *reference-row* rule.

    1.  Two rows can only belong to the same case if they already agree on
        the hard keys stored in KEY_COLS.

    2.  Inside every such key-group the rows are split whenever the gap
        between consecutive '[today(oprettelse)]' values exceeds 7 days.

    3.  In each resulting time-segment we repeatedly
          • pick as REFERENCE the row that contains the largest number of
            non-missing values in the eight Årsag columns (ties → first row),
          • compare every remaining row with that reference:

                – if the candidate is NaN in a column ⇒ that column matches
                – if the candidate has a value while the reference is NaN
                  ⇒ ***mismatch***
                – if both hold values they must be identical, otherwise
                  ⇒ ***mismatch***

          • every candidate with *no* mismatching columns joins the current
            duplicate cluster; the others are processed again with a new
            reference.  Only clusters that end up with ≥ 2 rows are kept.

    4.  The returned frame contains all rows that belong to such clusters and
        adds the columns
            is_duplicate  (always True)
            dup_case_id   (consecutive integer id per duplicate case)
    """
    import pandas as pd

    DATE_COL   = '[today(oprettelse)]'
    KEY_COLS   = [
        '[søg postnummer(postalcode)]',
        '[mand(patient data)]',
        '[alder (år)(patient data)]',
        '[generisk navn(gift1)]',
        '[navn(gift1)]',
    ]
    CAUSE_COLS = [
        '[ulykke/uheld(årsag)]',
        '[leg(årsag)]',
        '[suicidal/affekt(årsag)]',
        '[misbrug(årsag)]',
        '[forveksling(årsag)]',
        '[omhældning(årsag)]',
        '[fejldosering(årsag)]',
        '[levnedsmiddel(årsag)]',
    ]
    MAX_GAP = 7        # days -------------------------------------------------

    # -----------------------------------------------------------------
    # helper – build duplicate clusters inside ONE time-segment
    # -----------------------------------------------------------------
    def _clusters_in_segment(seg):
        """
        seg: dataframe that already shares KEY_COLS and lies inside a
             ≤ 7-day time window
        returns: list of dataframes, each a duplicate cluster (≥2 rows)
        """
        clusters = []
        pool = seg.copy().sort_index()

        while not pool.empty:
            # --- pick reference row
            nn_count = pool[CAUSE_COLS].notna().sum(axis=1)
            ref_idx  = nn_count.idxmax()          # first index in case of ties
            ref      = pool.loc[ref_idx, CAUSE_COLS]

            # --- vectorised comparison with reference
            cand_vals = pool[CAUSE_COLS]
            # candidate has value & reference is NaN  -> mismatch
            cond1 = cand_vals.notna() & ref.isna()
            # both have value but differ             -> mismatch
            cond2 = cand_vals.notna() & ref.notna() & (cand_vals != ref)

            mism = (cond1 | cond2).any(axis=1)
            match_mask = ~mism

            cluster_df = pool[match_mask]
            if len(cluster_df) >= 2:
                clusters.append(cluster_df.copy())

            # keep only the rows that did *not* match for the next loop
            pool = pool[mism]

        return clusters

    # -----------------------------------------------------------------
    # main routine
    # -----------------------------------------------------------------
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')

    all_clusters = []

    # 1. hard key grouping
    for _, grp in df.groupby(KEY_COLS, sort=False, observed=True):
        grp = grp.sort_values(DATE_COL)

        # 2. split by time gaps
        gap = grp[DATE_COL].diff().dt.days.abs().fillna(MAX_GAP + 1)
        grp['__seg'] = (gap > MAX_GAP).cumsum()

        for _, seg in grp.groupby('__seg', sort=False):
            all_clusters.extend(_clusters_in_segment(seg.drop(columns='__seg')))

    # -----------------------------------------------------------------
    # assemble result
    # -----------------------------------------------------------------
    if not all_clusters:    # no duplicates at all
        return pd.DataFrame(
            columns=df.columns.tolist() + ['is_duplicate', 'dup_case_id']
        )

    clusters_with_id = []
    for cid, cl in enumerate(all_clusters):
        cl = cl.copy()
        cl['is_duplicate'] = True
        cl['dup_case_id']  = cid
        clusters_with_id.append(cl)

    return pd.concat(clusters_with_id)

young_dups = find_duplicates(young)
old_dups   = find_duplicates(old)


In [12]:
print("Young sheet – duplicate cases:", young_dups['dup_case_id'].nunique())
print("Old   sheet – duplicate cases:", old_dups['dup_case_id'].nunique())


Young sheet – duplicate cases: 5886
Old   sheet – duplicate cases: 399


In [13]:
from IPython.display import display, HTML
import pandas as pd

def display_duplicate_cases(df: pd.DataFrame,
                            sheet_label: str,
                            n_cases: int = 20) -> None:
    """
    Show every column of every row that belongs to the first *n_cases*
    duplicate cases found in *df* (use n_cases=None to show them all).

    Parameters
    ----------
    df          The dataframe returned by `find_duplicates`
    sheet_label A title that will be shown above each block
    n_cases     Number of duplicate cases to display (None = all)
    """
    if df.empty:
        display(HTML(f"<h3>{sheet_label}: no duplicate cases</h3>"))
        return

    # make sure nothing is truncated in the HTML rendering
    with pd.option_context("display.max_columns", None,
                           "display.max_colwidth", None,
                           "display.width", None):
        case_ids = df["dup_case_id"].drop_duplicates()
        if n_cases is not None:
            case_ids = case_ids.head(n_cases)

        for cid in case_ids:
            subset = (
                df[df["dup_case_id"] == cid]
                .sort_values(DATE_COL)          # chronological inside the case
            )
            display(HTML(
                f"<h4>{sheet_label} – duplicate case {cid} "
                f"({len(subset)} rows)</h4>"
            ))
            display(subset)

# How many cases were detected?
print("Young sheet – duplicate cases:", young_dups["dup_case_id"].nunique())
print("Old   sheet – duplicate cases:", old_dups["dup_case_id"].nunique())

# Show the first 20 duplicate cases from each sheet
display_duplicate_cases(young_dups, "YOUNG SHEET", n_cases=20)
display_duplicate_cases(old_dups,   "OLD SHEET",   n_cases=20)

Young sheet – duplicate cases: 5886
Old   sheet – duplicate cases: 399


,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
26,Risiko for begrænset forgiftning,Cocain,NaN,15,2021-12-28 18:37:02,NaN,Indlæggelse,Psykiatriske sygdomme,Akut forgiftning,RUS,"[[STARTINFO]]rlar0142, 1-1-2022 18:50:50[[ENDINFO]][[STARTTEXT]]Man kan aldrig vide hvor rent et stof er, eller hvad det i øvrigt kan være blandet med. Lille risiko for at der kan være ætsningsskader, kokain kan også virke lokalbedøvende på tunge/ i mund. Anbefaler dyrkning fra belægninger (i forhold til om det er svampe-infektion), og gerne tilsyn ØNH-læge, da der er så udtalte gener. Noget sub-febril. Gerne obs EKG/syre-base-status i forhold til det forløb, der har fundet sted. Har der været brystsmerter, da koronarenzymer. Kan have haft markant sympatomimetisk respons på det ret høje kokain-indtag. Symptomatisk understøttende behandling. [[ENDTEXT]]803273[[STARTINFO]]rlar0142, 2-1-2022 11:10:47[[ENDINFO]][[STARTTEXT]]2. Opkald Præcis som ovenstående. Ved ætsningsskade, da tilrådes gerne scopi indenfor de første 12-24 timer i forhold til ruptur-risiko. Nu mange dage siden. Forsigtighed og symptomatisk behandling.[[ENDTEXT]]803366",2022-01-01 18:26:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hvidovre Hospital,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sygehus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[STARTINFO]]rlar0142, 1-1-2022 18:50:50[[ENDINFO]][[STARTTEXT]]Modtaget ovenstående, der for 4

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
63896,Risiko for begrænset forgiftning,Ukendt rusmiddel,NaN,NaN,2021-11-18 22:00:04,NaN,Ingen,NaN,Akut forgiftning,RUS,"[[STARTINFO]]cdit0001, 19-11-2021 16:07:02[[ENDINFO]][[STARTTEXT]]Melde det til politiet.[[ENDTEXT]]793826",2021-11-19 15:54:52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kvinde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Borger,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[STARTINFO]]cdit0001, 19-11-2021 16:07:02[[ENDINFO]][[STARTTEXT]]Har været ude i går og kan ikke huske noget fra kl 22. Ikke drukket meget og hun har helt blackout fra aftenen. Har ikke sin taske.[[ENDTEXT]]793826",2100,København Ø,19.0,"Noget i drink, Deprimerende",NaN,NaN,NaN,NaN,NaN,Omhældning,NaN,NaN,True,1
63985,Risiko for begrænset forgiftning,Ukendt rusmiddel,NaN,NaN,2021-11-19 23:08:49,NaN,Ingen,NaN,Akut forgiftning,RUS,"[[STARTINFO]]cdit0001, 20-11-2021 16:15:59[[ENDINFO]][[STARTTEXT]]Anbefaler at hun melder det til politi. Kan ikke få det opklaret. [[ENDTEXT]]794036",2021-11-20 16:02:16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kvinde,NaN,NaN,NaN,NaN,NaN,NaN,N

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
37,Risiko for livstruende forgiftning,Ethanol,NaN,NaN,2022-01-01 19:30:21,NaN,Indlæggelse,NaN,Akut forgiftning,DIVERSE,"[[STARTINFO]]anel0004, 1-1-2022 21:18:05[[ENDINFO]][[STARTTEXT]]Toxiak dosis depot > 4mg/kg; Alm tbl > 3mg/kg. Symptomer takykardi, svimmelhed, agitation, brystsmerter, mavesmerter, muskelrykninger og tremor, kramper og hypertermi. På hospital til behandling med kul og til symptomatisk behandling med fokus på respiration, cirkulation og kramper.[[ENDTEXT]]803299[[STARTINFO]]rlar0142, 1-1-2022 22:32:34[[ENDINFO]][[STARTTEXT]]2. Opkald Som ovenstående. Har taget solid toksisk dosis og depotformuleret. Fint med kul nu, gentag også om 2 timer atter ½ portion kul. Risiko er overstimulation med muskeluro, agitation evt. kramper. Ved symptomer, da Diazepam 5-10 mg iv, der efter 2 minutter kan gentages til respons. Tæt obs af vitalparametre. Ad EKG : QTc forlængelse ikke helt typisk for Methylphenidat-indtag. Gentag EKG og ved fortsat påvirkning, da telemetri. Ved brystsmerter coronarenzymer. Syrebase-status obs. Ved muskeluro/kramper, da rhabdomyolyse-tal. Obs i 12 timer. Alkohol promille kan dæmpe overstimulation. Velkommen til at kontakte GL ved yderligere. [[ENDTEXT]]803308[[STARTINFO]]rlar0142, 2-1-2022 10:52:16[[ENDINFO]][[STARTTEXT]]3. Opkald. Nu > 12 timer fra indtag. Kan fortsætte obs i psyk. regi. [[ENDTEXT]]803363",2022-01-01 21:00:35,METHYLPHENIDATHYDROCHLORID,NaN,mg,NaN,NaN,NaN,METHYLPHENIDATHYDROCHLORID,27,29,783,ZZMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
42,Uafklaret risiko,LEVOTHYROXINNATRIUM,NaN,NaN,2022-01-01 22:26:13,NaN,Indlæggelse,NaN,Akut forgiftning,MED,"[[STARTINFO]]anel0004, 1-1-2022 22:32:08[[ENDINFO]][[STARTTEXT]]Toxisk dosis Mere end eller lig 2000 mikrogram. Symptomdebut generelt efter 24 timer og varighed er ca 12-48 timer. Symptomer Tremor, varmefornemmelse/svedtendens, hypertermi, Ændret mentaltilstand også agitation, kramper, svær hypertension og arytmier. Kul kan gives op til 4 timer efter indtag. Vurdere om det er en akut eller kronisk forgiftning, obs blandingsforgiftninger feks BZD.[[ENDTEXT]]803310",2022-01-01 22:18:13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kvinde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Øvrige sundhedsvæsen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[STARTINFO]]anel0004, 1-1-2022 22:32:08[[ENDINFO]][[STARTTEXT]]Ambulance på vej til ovenstående. Har fået meldt, borger har indtaget Eltroxin kender ikke dosis eller tid. Borger skulle være agiteret og have låst sig inde. Hvad er toxisk dosis og symptomer?[[ENDTEXT]]803310",2620,Albertslund,25.0,Eltroxin,NaN,NaN,Suicidal/Affekt,NaN,NaN,NaN,NaN,NaN,True,3
52,Risiko for manifest forgiftning,LEVOTHYROXINNATRIUM,50,100,2022-01-01 21:45:53,5000,Indlæggelse,Psykiatriske sygdomme,Akut forgiftning,MED,"[[STARTINFO]]mhen0054, 2-1-2022 01:26:41[[ENDINFO]][[STARTTEXT]]Mild/moderat forgiftning. I

...

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
329,Risiko for manifest forgiftning,"Caffein, CODEINPHOSPHATHEMIHYDRAT, Magnesiumhydrox",250,80,2022-03-07 11:15:57,20000,Indlæggelse,NaN,Akut forgiftning,MED,"[[STARTINFO]]sgre0034, 7-3-2022 11:29:28[[ENDINFO]][[STARTTEXT]]OBS resp. , bevidsthedsniveau og vitale værdier. Skal indlægges til behandling med kul [[ENDTEXT]]817870[[STARTINFO]]sgre0034, 7-3-2022 11:57:42[[ENDINFO]][[STARTTEXT]]Risiko for salicylat forgiftning. Der skal tages en s-salicylat. Konc. >2,5 mmol(l indikation for alkalinisering af urin Instruks gennemgås Behandles med 2 port kul, med anæstesien i sonde, kan deles i små port så der gives 25 g x4 med 1 time i mellem. Indtagelsestidspunktet er ukendt, kan være indtaget i går. Der skal gives Phytomenadion (vitamin K) 10 mg i.v, KAD Væske, sørge for TD 1 ml/kg/time Behandles med BZD ved kramper Syre-base status, OBS metabolisk acidose, myoglobin og CK, Biokemi iht instruks som mailes til spøreger Ny henvendelse ved yderligere eller når der er svar på s-salicylat[[ENDTEXT]]817887[[STARTINFO]]sgre0034, 7-3-2022 13:31:36[[ENDINFO]][[STARTTEXT]]videregivet til sboe[[ENDTEXT]]817919[[STARTINFO]]sboe0017, 7-3-2022 23:55:29[[ENDINFO]][[STARTTEXT]]kl 13.30 Tilbagekald til spørger, Læge Elisabeth Længere samtale. Ingen nye oplysninger om et større indtag af medicin, men er noget forvirret og er udover det anførte også i behanlding med malfin og tramadol. Pt. er kendt med moderat KOL (BE 6 til den anførte ABG). CRP 17, men iøvrigt ikke tydelig tegn til infektion. Bedring efter katerisation. Bør udelukke andre årsager til bevidsthedpåvirkning, men en overdosering i går kan ikke udelukkes

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
352,Uafklaret risiko,Ukendt,NaN,NaN,2022-03-09 18:37:17,NaN,Intensiv afdeling,NaN,Akut forgiftning,RUS,"[[STARTINFO]]rlar0142, 10-3-2022 11:55:35[[ENDINFO]][[STARTTEXT]]Sag konf eped0037, fsoe0011 samt dpal0013 - videregivet til dpal0013, der kontakter indringer. [[ENDTEXT]]818608[[STARTINFO]]dpal0013, 10-3-2022 13:24:31[[ENDINFO]][[STARTTEXT]]Vurdering: Metabolisk acidose, må skyldes tilførsel af syre (ingen tegn på ætsning), tab af meget base (ikke mistanke) eller produktion af syre. Klinik med obs. beruselse og bevidsthedspåvirkning i hjemmet og derefter svær metabolisk acidose og aniongab 25 passer med indtag af toxisk alkohol, ethylenglycol. Usansdsynligt at det er metanol. Man plejer at se laktat stigning pga. fejlagtig måling af sure metablolitter - laktat normal her. Anbefaler: Kontakt til producnet af ABL-maskine, bliver metabolitter målt som laktat. Mikroskopi af urin, kovolutter oxalat krystaller CT-cerebrum kontrol, obs. begyndende hjerneødem. P-ethanol P-salicsylat Fomepizol (konf. med EBP, enig) Dialyse Ketoner 4: obs. hunger ketoner Ny kontakt ved behov og læge ringer tilbage for at høre om de skal give antidot da de starter dialyse op nu. Dette bekræftes og de vil skaffer Fomepizol. AC ethylenglycol er sendt til mail: soeren.jepsen@rsyd.dk[[ENDTEXT]]818619[[STARTINFO]]dpal0013, 11-3-2022 09:56:34[[ENDINFO]][[STARTTEXT]]Kontakt til intensiv læge Frank på tlf. 23800146. Man har givet Fomipozole bolus en dis vedligehold. Anbefaler at de giver i første gang to døgns behandling. Dosis hver 8. time nu hvor der køres CRRT (efter spørgsmål fra ita læge: det kan også gives som kontinuer

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
384,NaN,METHOTREXAT,NaN,NaN,2022-03-15 20:52:45,NaN,NaN,NaN,Kronisk forgiftning,ZZMED,"[[STARTINFO]]mann0002, 15-3-2022 20:53:39[[ENDINFO]][[STARTTEXT]]Ring igen når pt er ankommet mhp videre tiltag[[ENDTEXT]]819910[[STARTINFO]]mann0002, 15-3-2022 21:57:31[[ENDINFO]][[STARTTEXT]]2. opkald. Videregivet til bagvagt.(hhor0004). (spørger har vores instruks på Methotrexate)[[ENDTEXT]]819948[[STARTINFO]]hhor0004, 15-3-2022 22:27:50[[ENDINFO]][[STARTTEXT]]Endnu ingen blodprøvesvar. Opstarter antidot. Gennemgår typisk bivirkninger.[[ENDTEXT]]819958[[STARTINFO]]hhor0004, 16-3-2022 12:52:24[[ENDINFO]][[STARTTEXT]]Opfølgning via SP: endnu intet svar på MTX, normal knoglemarv, nyre- og levertal. I antidot behandling jvf instruks, lader til at være sluppet nådigt[[ENDTEXT]]820088[[STARTINFO]]nhen0047, 16-3-2022 13:03:40[[ENDINFO]][[STARTTEXT]]3. Opkald. Vider givet til KDAL0006[[ENDTEXT]]820092[[STARTINFO]]kdal0006, 16-3-2022 13:21:45[[ENDINFO]][[STARTTEXT]]Talt med BV Barbara Rubæk. Endnu ikke svar på de 2 x P-MTX taget tidligere. Er i calciumfolinat behandling ifølge vores vejleding 30 mg x 4 IV. Pt har som anført fejlagtigt fået 5 mg MTX dagligt de sidste 11 dage (skulle have haft 5 mg ugentligt). Egen læge har påbegyndt penicillin behandling 14/3 (800 mg x 3) pga. tonsillitis (muligvis betinget af MTX-associeret marvpåvirkning). Blodprøvestatus i dag: Hb 6,6, trombocyter 117, leukocyter 4, CRP 33, levertal normale. Bortset fra let hovedpine har pt det rimeligt. Ingen feber. Rådgivning: Fortsætter calciumfolinat uændret til svar på P-MTX. Bør skiftes til mere bredspektret AB. Ringer igen når P-MTX svar 

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
385,Risiko for manifest forgiftning,DIGOXIN,NaN,NaN,2022-03-15 19:34:08,NaN,Indlæggelse,NaN,Akut forgiftning,ZZMED,"[[STARTINFO]]anel0004, 15-3-2022 19:38:23[[ENDINFO]][[STARTTEXT]]Akut forgiftning: >2,6 nmol/l; Risiko for symptomgivende forgiftning. Konf. med HHOR0004. Anbefaler indlæggelse i aften, hvis bl.pr er taget korrekt, så ny kontrol af bl.pr kan tages i morgen samtidig med nyretal. Ved tilbagering: Bl.pr er taget korrekt, borger er svimmel og træt. Er ikke nået at undersøge yderligere men gør det. [[ENDTEXT]]819908",2022-03-15 19:24:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kvinde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vagtlæge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[STARTINFO]]anel0004, 15-3-2022 19:38:23[[ENDINFO]][[STARTTEXT]]Opkald fra lægevagt. Er på vej til ovenstående borger, der d.d. har fået målt en S-Digoxin på 3,7. Ved ikke hvordan borger har det. Hvad skal hun gøre? [[ENDTEXT]]819908",4550.0,Asnæs,86,DIGOXIN,Ulykke/uheld,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,18
388,Risiko for manifest forgiftning,DIGOXIN,62.5,2,2022-03-15 08:23:11,125,Indlæggelse,NaN,Akut forgiftning,ZZMED,"[[STARTINFO]]rlar0142, 16-3-2022 04:46:49[[ENDINFO]][[STARTTEXT]]Symptomatisk understøttende behandling i forhold til lungeødem. Ny s-Digoxin til morgen og pause Digoxin. Uvist om fejlmedicinering har fundet sted, obs ? S-Digox

,[ingen risiko(risiko)],[generisk navn(gift1)],[styrke(gift1)],[antal(gift1)],[indtag tid(gift1)],[dosis(gift1)],[ingen(anbefalet obs)],[psykiatriske sygdomme(komorbiditet)],[akut forgiftning(forespørgselsart)],[substans type(gift1)],[råd(tekst)],[today(oprettelse)],[navn(gift2)],[enhed(gift1)],[enhed(gift2)],[enhed(gift3)],[enhed(gift4)],[enhed(gift5)],[generisk navn(gift2)],[styrke(gift2)],[antal(gift2)],[dosis(gift2)],[substans type(gift2)],[søg hospital(hospital)],[navn(gift3)],[generisk navn(gift3)],[styrke(gift3)],[navn(gift6)],[generisk navn(gift6)],[styrke(gift6)],[antal(gift6)],[enhed(gift6)],[dosis(gift6)],[antal(gift3)],[substans type(gift6)],[navn(gift7)],[generisk navn(gift7)],[styrke(gift7)],[antal(gift7)],[enhed(gift7)],[dosis(gift7)],[substans type(gift7)],[navn(gift8)],[generisk navn(gift8)],[dosis(gift3)],[styrke(gift8)],[antal(gift8)],[enhed(gift8)],[dosis(gift8)],[substans type(gift8)],[navn(gift9)],[generisk navn(gift9)],[styrke(gift9)],[antal(gift9)],[enhed(gift9)],[dosis(gift9)],[substans type(gift9)],[substans type(gift3)],[navn(gift4)],[generisk navn(gift4)],[styrke(gift4)],[antal(gift4)],[dosis(gift4)],[substans type(gift4)],[navn(gift5)],[navn(gift10)],[generisk navn(gift10)],[styrke(gift10)],[antal(gift10)],[enhed(gift10)],[generisk navn(gift5)],[dosis(gift10)],[substans type(gift10)],[navn(gift11)],[generisk navn(gift11)],[styrke(gift11)],[antal(gift11)],[enhed(gift11)],[styrke(gift5)],[dosis(gift11)],[substans type(gift11)],[antal(gift5)],[navn(gift12)],[generisk navn(gift12)],[styrke(gift12)],[antal(gift12)],[enhed(gift12)],[dosis(gift12)],[substans type(gift12)],[dosis(gift5)],[navn(gift13)],[generisk navn(gift13)],[mand(patient data)],[styrke(gift13)],[antal(gift13)],[enhed(gift13)],[dosis(gift13)],[substans type(gift13)],[substans type(gift5)],[navn(gift14)],[generisk navn(gift14)],[styrke(gift14)],[antal(gift14)],[enhed(gift14)],[dosis(gift14)],[substans type(gift14)],[borger(spørger1)],[navn(gift15)],[generisk navn(gift15)],[styrke(gift15)],[antal(gift15)],[enhed(gift15)],[dosis(gift15)],[substans type(gift15)],[anamnese(tekst)],[søg postnummer(postalcode)],[søg by(postalcode)],[alder (år)(patient data)],[navn(gift1)],[ulykke/uheld(årsag)],[leg(årsag)],[suicidal/affekt(årsag)],[misbrug(årsag)],[forveksling(årsag)],[omhældning(årsag)],[fejldosering(årsag)],[levnedsmiddel(årsag)],is_duplicate,dup_case_id
390,Risiko for manifest forgiftning,HYDROCHLORTHIAZID,25,5,2022-03-16 09:59:40,125,Indlæggelse,NaN,Akut forgiftning,ZZMED,"[[STARTINFO]]cdit0001, 16-3-2022 12:23:55[[ENDINFO]][[STARTTEXT]]Så hurtigt som muligt.[[ENDTEXT]]820078[[STARTINFO]]sgre0034, 16-3-2022 13:16:24[[ENDINFO]][[STARTTEXT]]videregivet til FSOE[[ENDTEXT]]820104[[STARTINFO]]fsoe0011, 16-3-2022 14:09:16[[ENDINFO]][[STARTTEXT]]Opkald til akutmediciner Jacob Gennemgang af lægemidlerne: Magnyl â€“ 375mg - ikke toksisk dosis Paracetamol â€“ 20g - toksisk dosis - NAC er opstartet Losartan hydrochlorthiazid â€“ 500 125mg - lige på grænsen til toksisk dosis Donepezil â€“ 50mg - toksisk dosis - kolinerge bivirkninger diarre, sveden, savlen, muskelsvaghed. agitation Amlodipin â€“ 50mg -toksisk dosis Nebivolol â€“ 25mg - toksisk dosis Kombinationen af calciumantagonist og betablokker er vi mest bekymrede for og det er den der giver hypotension, bradycardi og påvirkningen af hjertet. Råd: aktiv kul 2 portioner da ukendt indtagelsestidspunkt Bradycardi: calciumgluconat IV, telemetri, EKKO (kald kard), overvej atropin (hjælp af anæstesi), væske. Overvejer at kalde MAT-kald, da hun bliver tiltagende bardycard og muligvis har brug for at komme på semi-intensivt afsnit. Instrukser på calciumantagonist og betablokker mailes. Ring igen ved behov! Konfereret med KDAL [[ENDTEXT]]820116",2022-03-16 11:55:34,Pamol,NaN,NaN,NaN,mg,mg,PARACETAMOL,500,40,20000,MED,NaN,Hjertemagnyl,ACETYLSALICYLSYRE,75.0,Norvasc,AMLODIPINBESILAT,10.0,5.0,NaN,50.0,5,MED,"Nebivolol \Portfarma\""""",Nebivololhydrochlorid,5.0,5.0,NaN,25.0,MED,NaN,NaN,375,NaN,NaN,NaN,NaN,NaN,N